# VPC Operations with Boto3

Region: ap-south-1

In [ ]:
# create the ec2 client and shared values
import boto3
import ipaddress
from botocore.exceptions import ClientError

REGION = "ap-south-1"
ec2 = boto3.client("ec2", region_name=REGION)
vpc_cidr = "10.20.0.0/16"
subnet_cidr = "172.31.16.0/24"
availability_zone = "ap-south-1a"

vpc_id = None
subnet_id = None
igw_id = None
route_table_id = None
association_id = None

In [2]:
# create the vpc
response = ec2.create_vpc(
    CidrBlock=vpc_cidr,
)
print(response)

vpc_id = response["Vpc"]["VpcId"]

{'Vpc': {'OwnerId': '837925920700', 'InstanceTenancy': 'default', 'Ipv6CidrBlockAssociationSet': [], 'CidrBlockAssociationSet': [{'AssociationId': 'vpc-cidr-assoc-084a81a7aa295667b', 'CidrBlock': '10.20.0.0/16', 'CidrBlockState': {'State': 'associated'}}], 'IsDefault': False, 'VpcId': 'vpc-0d64366a0425bead0', 'State': 'pending', 'CidrBlock': '10.20.0.0/16', 'DhcpOptionsId': 'dopt-0fe87498beadeac5b'}, 'ResponseMetadata': {'RequestId': 'c75052b9-f246-4487-92b6-4b75416deb84', 'HTTPStatusCode': 200, 'HTTPHeaders': {'x-amzn-requestid': 'c75052b9-f246-4487-92b6-4b75416deb84', 'cache-control': 'no-cache, no-store', 'strict-transport-security': 'max-age=31536000; includeSubDomains', 'content-type': 'text/xml;charset=UTF-8', 'content-length': '694', 'date': 'Tue, 19 May 2026 10:11:32 GMT', 'server': 'AmazonEC2'}, 'RetryAttempts': 0}}


In [23]:
# create subnet, internet gateway, and route table
if vpc_id is None:
    response = ec2.create_vpc(
        CidrBlock=vpc_cidr,
    )
    print(response)

    vpc_id = response["Vpc"]["VpcId"]

vpc_description = ec2.describe_vpcs(VpcIds=[vpc_id])
vpc_cidr = vpc_description["Vpcs"][0]["CidrBlock"]

vpc_network = ipaddress.ip_network(vpc_cidr)
subnet_network = ipaddress.ip_network(subnet_cidr)

existing_subnets = ec2.describe_subnets(
    Filters=[{"Name": "vpc-id", "Values": [vpc_id]}]
)["Subnets"]
existing_networks = [ipaddress.ip_network(subnet["CidrBlock"]) for subnet in existing_subnets]

if not subnet_network.subnet_of(vpc_network) or any(
    subnet_network.overlaps(existing) for existing in existing_networks
):
    new_prefix = max(24, vpc_network.prefixlen)
    if new_prefix > 28:
        new_prefix = 28
    subnet_cidr = None
    for candidate in vpc_network.subnets(new_prefix=new_prefix):
        if not any(candidate.overlaps(existing) for existing in existing_networks):
            subnet_cidr = str(candidate)
            break
    if subnet_cidr is None:
        raise ValueError("no available subnet range in this vpc")
    print(f"using subnet_cidr: {subnet_cidr}")

response = ec2.create_subnet(
    VpcId=vpc_id,
    CidrBlock=subnet_cidr,
    AvailabilityZone=availability_zone,
)
print(response)
subnet_id = response["Subnet"]["SubnetId"]

response = ec2.create_internet_gateway()
print(response)
igw_id = response["InternetGateway"]["InternetGatewayId"]

response = ec2.attach_internet_gateway(
    InternetGatewayId=igw_id,
    VpcId=vpc_id,
)
print(response)

response = ec2.create_route_table(
    VpcId=vpc_id,
)
print(response)
route_table_id = response["RouteTable"]["RouteTableId"]

response = ec2.create_route(
    RouteTableId=route_table_id,
    DestinationCidrBlock="0.0.0.0/0",
    GatewayId=igw_id,
)
print(response)

response = ec2.associate_route_table(
    RouteTableId=route_table_id,
    SubnetId=subnet_id,
)
print(response)
association_id = response["AssociationId"]

using subnet_cidr: 10.20.0.0/24
{'Subnet': {'AvailabilityZoneId': 'aps1-az1', 'MapCustomerOwnedIpOnLaunch': False, 'OwnerId': '837925920700', 'AssignIpv6AddressOnCreation': False, 'Ipv6CidrBlockAssociationSet': [], 'SubnetArn': 'arn:aws:ec2:ap-south-1:837925920700:subnet/subnet-0726389525247c2f6', 'EnableDns64': False, 'Ipv6Native': False, 'PrivateDnsNameOptionsOnLaunch': {'HostnameType': 'ip-name', 'EnableResourceNameDnsARecord': False, 'EnableResourceNameDnsAAAARecord': False}, 'SubnetId': 'subnet-0726389525247c2f6', 'State': 'available', 'VpcId': 'vpc-0d64366a0425bead0', 'CidrBlock': '10.20.0.0/24', 'AvailableIpAddressCount': 251, 'AvailabilityZone': 'ap-south-1a', 'DefaultForAz': False, 'MapPublicIpOnLaunch': False}, 'ResponseMetadata': {'RequestId': 'fd875f47-fb6f-4792-82e1-7eac61ef05e8', 'HTTPStatusCode': 200, 'HTTPHeaders': {'x-amzn-requestid': 'fd875f47-fb6f-4792-82e1-7eac61ef05e8', 'cache-control': 'no-cache, no-store', 'strict-transport-security': 'max-age=31536000; includeSu

In [24]:
# describe the vpc resources
print(ec2.describe_vpcs(VpcIds=[vpc_id]))
print(ec2.describe_subnets(SubnetIds=[subnet_id]))
print(ec2.describe_route_tables(RouteTableIds=[route_table_id]))

{'Vpcs': [{'OwnerId': '837925920700', 'InstanceTenancy': 'default', 'CidrBlockAssociationSet': [{'AssociationId': 'vpc-cidr-assoc-084a81a7aa295667b', 'CidrBlock': '10.20.0.0/16', 'CidrBlockState': {'State': 'associated'}}], 'IsDefault': False, 'BlockPublicAccessStates': {'InternetGatewayBlockMode': 'off'}, 'VpcId': 'vpc-0d64366a0425bead0', 'State': 'available', 'CidrBlock': '10.20.0.0/16', 'DhcpOptionsId': 'dopt-0fe87498beadeac5b'}], 'ResponseMetadata': {'RequestId': 'bdec0e8e-2978-4226-beaa-f1b6dcbd67f1', 'HTTPStatusCode': 200, 'HTTPHeaders': {'x-amzn-requestid': 'bdec0e8e-2978-4226-beaa-f1b6dcbd67f1', 'cache-control': 'no-cache, no-store', 'strict-transport-security': 'max-age=31536000; includeSubDomains', 'content-type': 'text/xml;charset=UTF-8', 'content-length': '798', 'date': 'Tue, 19 May 2026 10:33:26 GMT', 'server': 'AmazonEC2'}, 'RetryAttempts': 0}}
{'Subnets': [{'AvailabilityZoneId': 'aps1-az1', 'MapCustomerOwnedIpOnLaunch': False, 'OwnerId': '837925920700', 'AssignIpv6Addres

In [25]:
# delete vpc resources
response = ec2.disassociate_route_table(
    AssociationId=association_id,
)
print(response)

response = ec2.delete_route_table(
    RouteTableId=route_table_id,
)
print(response)

response = ec2.detach_internet_gateway(
    InternetGatewayId=igw_id,
    VpcId=vpc_id,
)
print(response)

response = ec2.delete_internet_gateway(
    InternetGatewayId=igw_id,
)
print(response)

response = ec2.delete_subnet(
    SubnetId=subnet_id,
)
print(response)

response = ec2.delete_vpc(
    VpcId=vpc_id,
)
print(response)

{'ResponseMetadata': {'RequestId': 'fb4a05fe-553c-4958-9b62-54d474df0c01', 'HTTPStatusCode': 200, 'HTTPHeaders': {'x-amzn-requestid': 'fb4a05fe-553c-4958-9b62-54d474df0c01', 'cache-control': 'no-cache, no-store', 'strict-transport-security': 'max-age=31536000; includeSubDomains', 'content-type': 'text/xml;charset=UTF-8', 'content-length': '299', 'date': 'Tue, 19 May 2026 10:33:37 GMT', 'server': 'AmazonEC2'}, 'RetryAttempts': 0}}
{'ResponseMetadata': {'RequestId': '7d0b85d1-6b7e-4058-9005-472c1337d461', 'HTTPStatusCode': 200, 'HTTPHeaders': {'x-amzn-requestid': '7d0b85d1-6b7e-4058-9005-472c1337d461', 'cache-control': 'no-cache, no-store', 'strict-transport-security': 'max-age=31536000; includeSubDomains', 'content-type': 'text/xml;charset=UTF-8', 'content-length': '221', 'date': 'Tue, 19 May 2026 10:33:37 GMT', 'server': 'AmazonEC2'}, 'RetryAttempts': 0}}
{'ResponseMetadata': {'RequestId': '048ce9fd-de55-4549-97ab-6a2626abd329', 'HTTPStatusCode': 200, 'HTTPHeaders': {'x-amzn-requestid'